# Week 2 / Day 01
## Covered today
1. Connecting to Multiple Frontier Models with APIs (OpenAI, Claude, Gemini)
2. Testing GPT-5 Models with Reasoning Effort and Scaling Puzzles
3. Testing Claude, GPT-5, Gemini & DeepSeek on Brain Teasers
4. Local Models with Ollama, Native APIs, and OpenRouter Integration
5. LangChain vs LiteLLM: Choosing the Right LLM Framework
6. LLM vs LLM: Building Multi-Model Conversations with OpenAI & Claude

### Now we make all the required imports

In [1]:
import os
from dotenv import load_dotenv
import requests
from openai import OpenAI
from IPython.display import Markdown, display, update_display


### Now we load all the API Keys

In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

### Now check if all loaded keys exist and are as per the format

In [3]:
if openai_api_key:
    if openai_api_key.startswith("sk-"):
        print(f"OpenAI      : OK           (begins {openai_api_key[:7]}...)")
    else:
        print("OpenAI      : WRONG FORMAT (should start with 'sk-')")
else:
    print("OpenAI      : MISSING")


if anthropic_api_key:
    if anthropic_api_key.startswith("sk-ant-"):
        print(f"Anthropic   : OK           (begins {anthropic_api_key[:10]}...)")
    else:
        print("Anthropic   : WRONG FORMAT (should start with 'sk-ant-')")
else:
    print("Anthropic   : MISSING")


if google_api_key:
    if google_api_key.startswith("AQ.Ab"):
        print(f"Google      : OK           (begins {google_api_key[:5]}...)")
    else:
        print("Google      : WRONG FORMAT (should start with 'AQ.Ab' or 'AIza')")
else:
    print("Google      : MISSING")


if openrouter_api_key:
    if openrouter_api_key.startswith("sk-or-"):
        print(f"OpenRouter  : OK           (begins {openrouter_api_key[:8]}...)")
    else:
        print("OpenRouter  : WRONG FORMAT (should start with 'sk-or-')")
else:
    print("OpenRouter  : MISSING")

OpenAI      : OK           (begins sk-proj...)
Anthropic   : OK           (begins sk-ant-api...)
Google      : OK           (begins AQ.Ab...)
OpenRouter  : OK           (begins sk-or-v1...)


In [4]:
# create clients for each provider
# for openai, we simply use OpenAI, for others we need to specify the base url and key, while using openai api library
openai_client = OpenAI()

google_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'
anthropic_url = 'https://api.anthropic.com/v1/'
openrouter_url = 'https://openrouter.ai/api/v1'
ollama_url = 'http://127.0.0.1:11434/v1'

In [5]:
google_client = OpenAI(base_url=google_url, api_key=google_api_key)
anthropic_client = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)
openrouter_client = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama_client = OpenAI(base_url=ollama_url, api_key='Ollama')

In [30]:
# Now test if each is working
tell_a_joke = [
    {'role': 'user', 'content': 'Tell a joke for a student who is on a journey to become an LLM engineer.'}
]
response = openai_client.chat.completions.create(model='gpt-5-mini', messages=tell_a_joke) #type: ignore
print(response.choices[0].message.content)

"How does an aspiring LLM engineer propose? 'Will you be my context window for life? I promise to never exceed your token limit.' "


In [31]:
response = anthropic_client.chat.completions.create(model='claude-haiku-4-5-20251001', messages=tell_a_joke) #type: ignore
print(response.choices[0].message.content)

# Why did the LLM engineer bring a ladder to the interview?

Because they heard the job required **climbing the attention stack**! 🪜

---

But seriously, here's one that might hit different:

**A transformer walks into a bar.** The bartender asks, "What'll it be?" 

The transformer says, "I'll have whatever the last 2,000 customers ordered, weighted by how relevant each one seems to me."

The bartender replies, "So... the same thing?"

"Probably," says the transformer. "But it took me 12 layers of reasoning to figure that out." 🤖

---

Good luck on your journey! May your gradients flow smoothly and your losses converge quickly! 📉


In [32]:
response = google_client.chat.completions.create(model='gemini-3.5-flash-lite', messages=tell_a_joke) #type: ignore
print(response.choices[0].message.content)

Why did the aspiring LLM engineer break up with their partner?

Because every time they tried to have a meaningful conversation, their partner felt the need to **hallucinate** a dramatic backstory, refused to admit when they were wrong, and kept responding with: 

*"As an AI language model, I don’t have feelings, but if I did, it would be 42."*


In [33]:
response = ollama_client.chat.completions.create(model='gemma3:270m', messages=tell_a_joke) #type: ignore
print(response.choices[0].message.content)

Why did the AI build its own engine? 

Because it wanted to be clever! 



In [34]:
response = openrouter_client.chat.completions.create(model='nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free', messages=tell_a_joke) #type: ignore
print(response.choices[0].message.content)

Why did the aspiring LLM engineer bring a ladder to the neural‑network party?

Because they heard the model’s loss was *going down* and wanted to **reach the next layer**! 😄


### Training vs Inference Scaling

Next up, we will have an easy puzzle and we will send this as a prompt to small models, with reasoning. We shall increase the reasoning and also add a bigger model to check if the answers are correct and at what level.

In [35]:
easy_puzzle = [
    {'role': 'user', 'content': 
    "You toss 2 coins. One of them is heads. What's the probablity that the other is tails. Answer with probablity only."}
]

In [36]:
response = openai_client.chat.completions.create(model='gpt-5-nano', messages=easy_puzzle, reasoning_effort='minimal')
print(response.choices[0].message.content)

1/3


This above is a wrong answer. Now, we increase the reasoning to low.

In [37]:
response = openai_client.chat.completions.create(model='gpt-5-nano', messages=easy_puzzle, reasoning_effort='low')
print(response.choices[0].message.content)

2/3


Now, we can see that with increase in reasoning, the answer becomes correct. Now, with minimal reasoning, we bump up the model from nano to mini

In [38]:
response = openai_client.chat.completions.create(model='gpt-5-mini', messages=easy_puzzle, reasoning_effort='minimal')
print(response.choices[0].message.content)

2/3


With now, even the minimal reasoning, but a bumped up model, correct answer was received.

### Nextup we work with Anthropic and Google Client Libraries

We start with google, by first installing -U google-genai and then importing genai from google

In [39]:
from google import genai

In [40]:
client = genai.Client()
response = client.interactions.create(
    model='gemini-3.5-flash-lite',
    input='Describe color blue who has never been able to see.'
)
print(response.output_text)

How do you explain a color to someone who has never seen light? You cannot use the eyes; instead, you must translate sight into the languages the body already knows: touch, temperature, emotion, and sound. 

If blue were a feeling, it would be the quietest part of the night. It is the sensation of absolute stillness—not an empty quiet, but a deep, breathing calm, like the feeling of being completely alone in a vast, open room where the air is cool and still. 

Imagine taking a deep breath of winter air, high up in the mountains, just before the sun rises. That crispness inside your lungs, that pure, clear cold—that is the essence of blue. 

It has a texture. It is not rough like tree bark or sharp like a thorn. Blue feels like smooth, cool water sliding over your skin, or the sleek, heavy weight of polished glass that has been sitting in the shade. It is the feeling of a gentle breeze moving across your face on a day that is neither hot nor cold, but perfectly balanced.

In sounds, blu

Next we move to anthropic, first we install anthropic, then we from anthropic, we import Anthropic

In [46]:
from anthropic import Anthropic
client = Anthropic()
response = client.messages.create(
    max_tokens=1000,
    model='claude-haiku-4-5-20251001',
    messages=easy_puzzle #type: ignore
)
print(response.content[0].text)

2/3


While Open AI SDK is used mostly, since they were the first to start the AI revolution and most people have moved from Open AI SDK, sometimes people can use SDKs of the providers they are using, hence, it is good to understand how their SDKs work.

### Next we move to Abstraction Layers (Langchain & LiteLLM)